# 3 · A full decision model with Unsloth

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nipunbatra/system-one/blob/main/notebooks/03_unsloth_decision_model.ipynb)

Companion to the lecture **System 1 models: deciding without writing**.

Notebook 2 trained a head on a frozen LLM for one fixed question. Unsloth's `FastDecisionModel` does the general version: LoRA adapters on the LLM plus a small head that reads **every question about an input in one pass** and scores each option, for the three question types `choice`, `noul` and `score`.

This notebook follows Unsloth's official decision-model notebook ([guide](https://unsloth.ai/docs/basics/train-your-own-decision-model-with-unsloth), [original notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(4B)-Decision.ipynb)), with the smaller `unsloth/Qwen3.5-0.8B` so it fits a free T4.

**Needs a CUDA GPU**: Runtime → Change runtime type → T4. Unsloth reports about 42 minutes for a full run of the 0.8B model; the T4 trains Qwen3.5 in float32, so expect it to be slower there. Start it before class.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 1. An LLM plus a decision head

In [ ]:
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
import torch

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-0.8B",
    max_seq_length = 2048,   # longest input; long inputs keep the question and options
    load_in_4bit = True,     # 4-bit weights to save memory
)
model = FastDecisionModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0,   # LoRA: train small adapters, not the whole LLM
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 2. Data: state, questions, gold

Each row has a `state` (the input), the `questions` to decide about it, and the `gold` answers.

In [ ]:
from datasets import load_dataset
import json

dataset = load_dataset("LocalLLaMA/typed-decisions", "all", split = "train")
print(dataset[0]["state"][:500])
print(json.dumps(json.loads(dataset[0]["questions"]), indent = 2)[:1000])
print(dataset[0]["gold"])

In [ ]:
items, report = FastDecisionModel.build_dataset(dataset, tokenizer, model)
print(f"Skipped {report['skipped']} of {report['total']} decisions")

train_items, eval_items = FastDecisionModel.split_holdout(items, seed = 3407)   # held-out rows for calibration
print(f"{len(train_items)} training rows, {len(eval_items)} held out")

Accuracy before training. The head is new, so this is close to guessing.

In [ ]:
FastDecisionModel.evaluate(model, tokenizer, eval_items)

## 3. Train

In [ ]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_items,
    eval_dataset = eval_items,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,                 # a short run; use num_train_epochs = 2 for the full one
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        eval_strategy = "epoch",
        logging_steps = 10,
        output_dir = "outputs",
        report_to = "none",
        seed = 3407,
    ),
)
trainer_stats = trainer.train()
print(round(trainer_stats.metrics["train_runtime"] / 60, 1), "minutes")

## 4. Calibrate on the held-out rows, then test

`calibrate` adjusts the probabilities on rows the model did not train on, and prints held-out accuracy, the calibration error (`ece`) and the loss.

In [ ]:
FastDecisionModel.calibrate(model, tokenizer, eval_items)

In [ ]:
test = load_dataset("LocalLLaMA/typed-decisions", "all", split = "test")
test_items, _ = FastDecisionModel.build_dataset(test, tokenizer, model)
FastDecisionModel.evaluate(model, tokenizer, test_items)

## 5. Decide

One call, three questions about one email, a probability for every option.

In [ ]:
FastDecisionModel.for_inference(model)
answers = FastDecisionModel.predict(
    model,
    tokenizer,
    "Hi, I was charged twice for invoice #4411. Please refund the duplicate today.",
    {
        "team": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "billing": "invoices, payments, refunds",
                "technical": "bugs, outages, errors",
                "sales": "pricing, new plans",
            },
        },
        "refund": {"type": "noul", "instructions": "Does the customer ask for a refund?"},
        "urgency": {
            "type": "score",
            "instructions": "How urgent is this?",
            "criteria": ["not urgent", "soon", "today"],
        },
    },
)
for name, result in answers.items():
    print(name, result["answer"], {k: round(v, 3) for k, v in result["probabilities"].items()})

In [ ]:
model.save_pretrained("qwen_decisions")   # LoRA adapters, the decision head and its calibration